In [ ]:
# -------------------------------------------------
#  Imports & configuration
# -------------------------------------------------
import pathlib
import os
import datetime as dt
from typing import List, Sequence, Dict, Any

import pandas as pd
import requests

# --------------------------------------------------------------

# --------------------------------------------------------------
# Azure ADLS imports
# --------------------------------------------------------------
from azure.storage.filedatalake import DataLakeServiceClient
from pathlib import PurePosixPath
import io
# --------------------------------------------------------------


In [ ]:
# -------------------------------------------------
#  variables
# -------------------------------------------------


HA_URL = 'http://10.44.61.150:8123'

HEADERS = "bearer token"

STATISTIC_ID = 'sensor.<>'
START = dt.datetime(2026, 2, 1, tzinfo=dt.timezone.utc)
END = dt.datetime(2026, 2, 28, tzinfo=dt.timezone.utc)
PERIOD="hour"
TYPES=['state',]

In [ ]:
# ------------------------------------------------------------------
# 1. Helper functions – thin wrapper around the recorder service
# ------------------------------------------------------------------
def _post_service(domain: str, service: str, payload: dict) -> dict:
    """Call a Home‑Assistant service via the REST API."""
    url = f"{HA_URL}/api/services/{domain}/{service}?return_response"
    resp = requests.post(url, headers=HEADERS, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()


# ------------------------------------------------------------------
# 2. Helper: Get_statistics from Home Assistant API
# ------------------------------------------------------------------
def _get_statistics(
    statistic_id: str,
    start: dt.datetime,
    end: dt.datetime,
    period: str = "hour",
    types: Sequence[str] = ("state",),
) -> pd.DataFrame:
    """
    Retrieve statistics through the ``recorder.get_statistics`` service.
    (unchanged – see your original implementation)
    """
    # Normalise the ``types`` argument – allow any iterable (list, tuple, set,…)
    if not isinstance(types, (list, tuple, set)):
        raise TypeError("`types` must be a sequence of strings.")
    type_list: List[str] = list(types)

    payload = {
        "statistic_ids": [statistic_id],
        "start_time": start.isoformat(),
        "end_time": end.isoformat(),
        "period": period,
        "types": type_list,
    }

    # -----------------------------------------------------------------
    # 1️⃣  Call the recorder service
    # -----------------------------------------------------------------
    raw = _post_service("recorder", "get_statistics", payload)

    # -----------------------------------------------------------------
    # 2️⃣  Detect an error string early
    # -----------------------------------------------------------------
    if isinstance(raw, str):
        raise RuntimeError(
            f"Recorder service returned an error string: {raw!r}. "
            "Check that the `statistic_id` exists and that the payload is correct."
        )

    # -----------------------------------------------------------------
    # 3️⃣  Extract the nested statistics dictionary
    # -----------------------------------------------------------------
    try:
        stats_dict = raw["service_response"]["statistics"]
    except (KeyError, TypeError) as exc:
        raise RuntimeError(
            f"Unexpected recorder response format: {raw!r}. "
            "Expected keys 'service_response' → 'statistics'."
        ) from exc

    # -----------------------------------------------------------------
    # 4️⃣  Pull the list of points for the requested statistic_id
    # -----------------------------------------------------------------
    points = stats_dict.get(statistic_id, [])
    if not isinstance(points, list):
        raise RuntimeError(
            f"Statistic data for '{statistic_id}' is not a list: {points!r}"
        )

        # -----------------------------------------------------------------
    # 5️⃣  Convert each point into a row for the DataFrame
    # -----------------------------------------------------------------
    rows = []
    for point in points:
        # Timestamp – we only need the start of the bucket
        ts_str = point.get("start")
        if not ts_str:
            continue
        ts = (
            dt.datetime.fromisoformat(ts_str.rstrip("Z"))
        )

        # Build a dict that always contains the timestamp and then each
        # requested type, normalising unavailable values to None.
        row = {"start": ts}
        for typ in type_list:
            raw_val = point.get(typ)

            # ``state`` can be a string (e.g. "on"/"off") – keep it as‑is.
            # All numeric‑looking types are coerced to float when possible.
            if typ == "state":
                row[typ] = raw_val
                continue

            try:
                row[typ] = (
                    float(raw_val)
                    if raw_val not in (None, "unavailable", "unknown")
                    else None
                )
            except (ValueError, TypeError):
                row[typ] = None

        rows.append(row)

    # -----------------------------------------------------------------
    # 6️⃣  Sort by timestamp and build the DataFrame
    # -----------------------------------------------------------------
    rows.sort(key=lambda r: r["start"])
    df = pd.DataFrame(rows)

    # Ensure the column order is deterministic: start first, then the types
    column_order = ["start"] + type_list
    df = df[column_order]

    return df


# ------------------------------------------------------------------
# 3. Azure Data Lake client factory
# ------------------------------------------------------------------
def _adls_client_from_connection_string(conn_str: str) -> DataLakeServiceClient:
    """
    Build a DataLakeServiceClient from a connection string.
    You can also use a SAS token or Azure AD credentials – adapt as needed.
    """
    return DataLakeServiceClient.from_connection_string(conn_str)


def store_monthly_parquet(
        df: pd.DataFrame, 
        adls_client, 
        filesystem, 
        year: int, 
        month: int
    ):
    # Ensure the DataFrame has an `entity_id` column for later filtering
    # df = df[['entity_id', 'timestamp', ...]]

    # Build the path: home_assistant_stats/2024/2024-01.parquet
    filename = f"{year}-{month:02d}.parquet"
    path = f"home_assistant_stats/{year}/{filename}"

    # Convert to Parquet in memory
    buf = io.BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow", compression="snappy")
    buf.seek(0)

    # Upload (overwrite if the month is being regenerated)
    file_client = adls_client.get_file_client(filesystem, path)
    file_client.upload_data(buf.read(), overwrite=True)

In [ ]:
#  Load DeltaInzicht CSVs (one‑liner per year)
# -------------------------------------------------
DATA_ROOT = pathlib.Path("EnergyLogs")
YEARS = [2020, 2021, 2022]

# Columns we actually need (0‑based indices from the original script)
COLS_TO_USE = [3, 4, 5, 6, 7, 8]

def read_delta(year: int) -> pd.DataFrame:
    """Read a single year's CSV and return a tidy DataFrame."""
    fp = DATA_ROOT / f"EnergyLogDevices_{year}.csv"
    df = pd.read_csv(
        fp,
        delimiter=";",
        header=0,
        decimal=",",
        usecols=COLS_TO_USE,
        # give the columns explicit names – makes later code clearer
        names=["Apparaat", "Datum", "Uur", "Meetwaarde", "Eenheid", "Kosten"]
    )
    # Build a proper datetime column (hour‑only resolution)
    df["DateTime"] = pd.to_datetime(
        df["Datum"] + " " + df["Uur"].str[:2] + ":00",
        format="%Y-%m-%d %H:%M",  # ISO‑like, but safe
        errors="coerce",
    )
    # Drop the raw date/time columns we no longer need
    return df.drop(columns=["Datum", "Uur"])

# Read all years in one go and concatenate
df_delta = pd.concat([read_delta(y) for y in YEARS], ignore_index=True)

print("\n✅ CSV‑import voltooid")
print("Beschikbare jaren :", df_delta['DateTime'].dt.year.sort_values().unique())
print("Aantal rijen    :", len(df_delta))

# Optional sanity check
df_delta.loc[df_delta['Apparaat'].str.contains('Produced', na=False), 'Meetwaarde'] *= -1
df_delta['month'] = df_delta["DateTime"].dt.strftime("%Y-%m")
df_delta.set_index('month', inplace=True)
df_archive = df_delta.pivot_table(
    index=['month'], 
    values=['Meetwaarde'],
    columns = 'Apparaat',
    aggfunc='sum', 
    margins=False
    )
# ------- verwijder het 'value' column level -------
df_archive.columns = df_archive.columns.droplevel(0)
df_archive.rename(columns={
    'gas Meter': 'gas_import',
    'Low Tariff Consumed': 'kwh_import_low',
    'Normal Tariff Consumed': 'kwh_import_normal',
    'Low Tariff Produced': 'kwh_export_low',
    'Normal Tariff Produced': 'kwh_export_normal',
}, inplace=True)
df_archive.tail(10)

In [ ]:
#df = _get_statistics(
#    statistic_id=statistic_id,
#    start=start,
#    end=end,
#    period=period,
#    types=types
#)

ADLS_CONN = _adls_client_from_connection_string(os.getenv("ADLS_CONNECTION_STRING"))

# Assuming the original HA helpers (HA_URL, HEADERS, etc.) are already defined
store_monthly_parquet(
        df = df_archive,
        adls_client=ADLS_CONN,
        filesystem="homelab-datalake", 
        year = YEARS[0],
        month = 1
)

